# 第 11 章 · LoRA 低秩适配

> 📄 配套动画网页：`../11_lora.html`　（先用浏览器打开，把原理动画看一遍再来写代码）

**所属部分**：第三部分 · 训练

想给模型加一项新技能，不必重训全部参数。LoRA 冻结底座，只训练一条低秩小分支。

---


## 🎯 本章目标

- 理解低秩分解为什么有效
- 掌握 `setattr` 动态挂模块、`named_modules` 遍历
- 理解 A/B 的初始化策略
- 能从零实现 `LoRA`、`apply_lora` 与权重合并

---


## 🖼 官方配图

![LoRA 挂在这些层上：架构里所有 in_features == out_features 的方阵线性层（Q/K/V/O 投影、FFN 三投影）](../images/LLM-structure.jpg)

*LoRA 挂在这些层上：架构里所有 in_features == out_features 的方阵线性层（Q/K/V/O 投影、FFN 三投影）*

---


## 📖 背景：只训练 1% 的参数

```python
h = W @ x  +  (α/r) · B @ A @ x
      ↑              ↑
   冻结不变      只训练这两个小矩阵

A: (in_features, r)   高斯初始化
B: (r, out_features)  零初始化 → 初始时 B@A = 0，不破坏原模型
```

为什么有效？因为微调时权重的**变化量 ΔW 本身就是低秩的**。
用一个 `d×r` 加一个 `r×d` 的小矩阵（r 远小于 d）就足够表达。

**B 为什么零初始化**：这样训练开始时 `W + B@A = W`，模型行为与原来完全一致，
不会有「一开始就被随机扰动」的问题。

---



# 第一部分 · 基础函数演示

> 这一部分**可以直接运行**。先把本章要用到的 Python / PyTorch 函数和方法摸熟，后面自己写的时候才不会卡在 API 上。


### 11.1 `nn.Linear` 的 `in_features` / `out_features`

In [ ]:
import torch
from torch import nn

lin = nn.Linear(768, 768, bias=False)
print("in_features :", lin.in_features)
print("out_features:", lin.out_features)
print("weight 形状 :", lin.weight.shape)
print("参数量      :", lin.weight.numel())

### 11.2 低秩分解：`B @ A` 的形状

`A` 把 768 维压到 r 维，`B` 再把它升回 768 维。

In [ ]:
d, r = 768, 16
A = nn.Linear(d, r, bias=False)      # [16, 768]
B = nn.Linear(r, d, bias=False)      # [768, 16]

x = torch.randn(2, d)
print("x          :", x.shape)
print("A(x)       :", A(x).shape)
print("B(A(x))    :", B(A(x)).shape)
print("增量矩阵形状:", (B.weight @ A.weight).shape)   # [768, 768]
print("LoRA 参数量 :", A.weight.numel() + B.weight.numel())
print("全参数量    :", d * d)

### 11.3 初始化：A 高斯、B 零

这是 LoRA 能「无损挂载」的关键。

In [ ]:
A = nn.Linear(768, 16, bias=False)
B = nn.Linear(16, 768, bias=False)
A.weight.data.normal_(mean=0.0, std=0.02)
B.weight.data.zero_()

x = torch.randn(2, 768)
delta = B(A(x))
print("B@A 初始输出是否全为 0:", torch.allclose(delta, torch.zeros_like(delta)))

### 11.4 `setattr` 动态挂模块

LoRA 要在运行时给已有的 `nn.Linear` 挂一个 `lora` 子模块。

In [ ]:
class Holder(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Linear(8, 8, bias=False)

h = Holder()
setattr(h.proj, "lora", nn.Linear(8, 2, bias=False))     # 动态挂载
print("挂上了吗:", hasattr(h.proj, "lora"))
print("会被注册为子模块:", "proj.lora" in [n for n, _ in h.named_modules()])

### 11.5 替换 forward：让原输出加上 LoRA 增量

用一个闭包保存原 forward，再替换掉。

In [ ]:
class Holder(nn.Module):
    def __init__(self):
        super().__init__()
        self.proj = nn.Linear(8, 8, bias=False)

h = Holder()
lora = nn.Linear(8, 8, bias=False)     # 简化：直接用满秩代替 A@B
setattr(h.proj, "lora", lora)

original_forward = h.proj.forward
def forward_with_lora(x, layer1=original_forward, layer2=lora):
    return layer1(x) + layer2(x)
h.proj.forward = forward_with_lora

x = torch.randn(2, 8)
print("原输出 + LoRA 增量:", h.proj(x).shape)
print("差值 == lora(x):", torch.allclose(h.proj(x) - original_forward(x), lora(x), atol=1e-5))

### 11.6 权重合并

推理时把 `B@A` 加回 `W`，就没有额外计算开销了。

In [ ]:
d, r = 8, 2
W = nn.Linear(d, d, bias=False)
A = nn.Linear(d, r, bias=False)
B = nn.Linear(r, d, bias=False)
A.weight.data.normal_(0, 0.1)

W_merged = W.weight.data.clone() + (B.weight.data @ A.weight.data)
print("合并前后形状一致:", W_merged.shape == W.weight.shape)
print("合并后参数总量不变:", W_merged.numel())

# 第二部分 · 动手实现（只给提示，留白给你）

> 下面每一节都**没有答案**。请根据文字提示 + 第一部分演示过的函数，自己从零写出来。
> 写不出来时，先回去看第一部分的演示，再对照网页里的动画想清楚数据形状。

LoRA 的三个部分都要自己写：类定义、挂载、合并。

> 💡 每个留白代码块后面都跟了一个**测试块**。先写代码，再运行测试块。


In [ ]:
# ════════════════════ 自测工具 ════════════════════
# 每个留白代码块后面都跟一个测试块。先在上面写代码，再运行测试块。
# 没写之前运行也不会报错，只会提示「未实现」。
import torch
import torch.nn.functional as F

_PASS, _FAIL, _SKIP = [], [], []

def check(name, fn):
    """运行一个测试。捕获异常并打印结果，不会中断后面的测试。"""
    try:
        fn()
        print(f"✅ 通过   {name}")
        _PASS.append(name)
    except NameError as e:
        print(f"⏸  未实现 {name} —— {e}")
        _SKIP.append(name)
    except AssertionError as e:
        print(f"❌ 失败   {name} —— {e}")
        _FAIL.append(name)
    except Exception as e:
        print(f"❌ 出错   {name} —— {type(e).__name__}: {e}")
        _FAIL.append(name)

def close(a, b, tol=1e-4, msg=""):
    """断言两个张量 / 数字足够接近"""
    a, b = torch.as_tensor(a).float().reshape(-1), torch.as_tensor(b).float().reshape(-1)
    assert a.numel() == b.numel(), f"元素个数不一致 {a.numel()} vs {b.numel()} {msg}"
    assert torch.allclose(a, b, atol=tol), f"数值不接近（容差 {tol}）{msg}"

def same(a, b, msg=""):
    """断言两个张量完全相同"""
    a, b = torch.as_tensor(a), torch.as_tensor(b)
    assert a.shape == b.shape, f"形状不一致 {tuple(a.shape)} vs {tuple(b.shape)} {msg}"
    assert torch.equal(a, b), f"内容不相同 {msg}"

def summary():
    print()
    print(f"通过 {len(_PASS)} · 失败 {len(_FAIL)} · 未实现 {len(_SKIP)}")
    if _FAIL: print("需要修的：", "、".join(_FAIL))
    if _SKIP: print("还没写的：", "、".join(_SKIP))
    if _PASS and not _FAIL and not _SKIP:
        print("本章全部通过 🎉")

print("自测工具已就绪 —— 每写完一个留白，就运行它下面的测试块。")

### 11.7 实现 `LoRA` 类

**要求**：
- `__init__(self, in_features, out_features, rank)`
- `self.A = nn.Linear(in_features, rank, bias=False)`，权重用 `normal_(0, 0.02)` 初始化
- `self.B = nn.Linear(rank, out_features, bias=False)`，权重**零初始化**
- `forward(x)` 返回 `B(A(x))`

In [ ]:
# ======== 你的实现：LoRA ========
#
# 【要求】
#   - 继承 nn.Module
#   - A: in_features -> rank（高斯初始化 std=0.02）
#   - B: rank -> out_features（零初始化）
#   - forward(x) = self.B(self.A(x))
#
# 【可用到的函数 / 方法】
#   - `nn.Linear(..., bias=False)`
#   - `Tensor.data.normal_` / `.zero_()`
#
# 提示：先想清楚输入输出的形状，再动手。
#
# ↓↓↓ 在这里写你的代码 ↓↓↓




In [ ]:
# ════════════ 测试：LoRA ════════════
# 先在上面把代码写出来，再运行这一格。
def _t():
    torch.manual_seed(0)
    lora = LoRA(16, 16, rank=4)
    assert tuple(lora.A.weight.shape) == (4, 16), f"A 形状应为 (4, 16)，得到 {tuple(lora.A.weight.shape)}"
    assert tuple(lora.B.weight.shape) == (16, 4), f"B 形状应为 (16, 4)，得到 {tuple(lora.B.weight.shape)}"
    close(lora(torch.randn(3, 16)), torch.zeros(3, 16), tol=1e-8,
          msg="B 零初始化时 LoRA 初始输出必须为 0，否则挂载会破坏原模型")
    n = sum(p.numel() for p in lora.parameters())
    assert n == 4 * 16 + 16 * 4, f"参数量应为 {4 * 16 + 16 * 4}，得到 {n}"

check("LoRA 形状 / 零初始化 / 参数量", _t)

### 11.8 实现 `apply_lora`

**要求**：遍历 `model.named_modules()`，对每个 `nn.Linear` 且 `in_features == out_features` 的层，
挂上 `LoRA`，并把它的 forward 替换成「原输出 + LoRA 输出」。

**注意**：闭包要用默认参数绑定 `original_forward` 和 `lora`，
否则循环变量会被后续迭代覆盖（经典的 Python 闭包陷阱）。

In [ ]:
# ======== 你的实现：apply_lora ========
#
# 【要求】
#   - 签名：apply_lora(model, rank=16)
#   - 遍历 model.named_modules()
#   - 判断 isinstance(module, nn.Linear) and module.in_features == module.out_features
#   - 创建 LoRA 并 setattr(module, 'lora', lora)
#   - 替换 module.forward 为 forward_with_lora(x, layer1=原forward, layer2=lora)
#
# 【可用到的函数 / 方法】
#   - `model.named_modules()`
#   - `setattr`
#   - 默认参数绑定闭包变量
#
# 提示：先想清楚输入输出的形状，再动手。
#
# ↓↓↓ 在这里写你的代码 ↓↓↓




In [ ]:
# ════════════ 测试：apply_lora ════════════
# 先在上面把代码写出来，再运行这一格。
def _t():
    torch.manual_seed(0)
    model = nn.Sequential(nn.Linear(8, 8, bias=False), nn.Linear(8, 4, bias=False))
    x = torch.randn(2, 8)
    before = model(x).detach().clone()

    apply_lora(model, rank=2)
    assert hasattr(model[0], "lora"), "in_features == out_features 的层应该被挂上 lora"
    assert not hasattr(model[1], "lora"), "in_features != out_features 的层不该被挂 lora"
    close(model(x), before, tol=1e-6,
          msg="挂载 LoRA 后原模型的输出变了 —— 检查 B 是不是零初始化")

    model(x).sum().backward()
    assert model[0].lora.A.weight.grad is not None, "LoRA 的 A 应该能收到梯度"

check("apply_lora 挂载 / 不改变原输出", _t)

### 11.9 实现权重合并 `merge_lora`

**要求**：对每个带 `.lora` 的 `nn.Linear`，执行 `W += B.weight @ A.weight`，
然后把所有 `.lora.` 相关的键从 state_dict 里剔除，保存到 `save_path`。

**提示**：`state_dict = {k: v for k, v in model.state_dict().items() if '.lora.' not in k}`

In [ ]:
# ======== 你的实现：merge_lora ========
#
# 【要求】
#   - 签名：merge_lora(model, save_path)
#   - 遍历 named_modules，找到有 .lora 属性的 nn.Linear
#   - module.weight.data += module.lora.B.weight.data @ module.lora.A.weight.data
#   - 导出 state_dict 时过滤掉 '.lora.' 键
#   - torch.save 到 save_path
#
# 【可用到的函数 / 方法】
#   - `torch.save`
#   - 字典推导式过滤键
#
# 提示：先想清楚输入输出的形状，再动手。
#
# ↓↓↓ 在这里写你的代码 ↓↓↓




In [ ]:
# ════════════ 测试：merge_lora ════════════
# 先在上面把代码写出来，再运行这一格。
def _t():
    import tempfile, os
    torch.manual_seed(0)
    model = nn.Sequential(nn.Linear(8, 8, bias=False))
    apply_lora(model, rank=2)
    with torch.no_grad():
        model[0].lora.A.weight.normal_(0, 0.1)
        model[0].lora.B.weight.normal_(0, 0.1)

    x = torch.randn(2, 8)
    with_lora = model(x).detach().clone()
    expected_W = (model[0].weight.detach()
                  + model[0].lora.B.weight.detach() @ model[0].lora.A.weight.detach())

    path = os.path.join(tempfile.gettempdir(), "mm_lora_merged.pth")
    merge_lora(model, path)
    sd = torch.load(path, map_location="cpu")
    assert not any(".lora." in k for k in sd), f"合并后的权重里不该再有 lora 参数：{list(sd.keys())}"
    merged_W = sd["0.weight"]
    close(merged_W, expected_W, tol=1e-4, msg="合并后的权重应等于 W + B@A")
    close(x @ merged_W.T, with_lora, tol=1e-4, msg="用合并权重算出的输出应与原来一致")

check("merge_lora 权重合并正确", _t)

In [ ]:
summary()

# 第三部分 · 自测与思考

- [ ] 能解释为什么 B 要零初始化
- [ ] 挂载后原模型的行为会变吗？（不应该变）
- [ ] 参数量降到多少？
- [ ] 合并后的权重与「W + BA」一致吗？

**思考题**：rank 越大效果一定越好吗？为什么？
